# Предсказание рейтинга SoFIFA на следующий год после сезона

Матч → трансформер (обученный энкодер) → эмбеддинг по каждому игроку в матче → **новая голова** предсказывает рейтинг игрока в **следующем году** после сезона (таргет из `dataset/sofifa_ratings_by_season.csv`).

In [1]:
import sys
from pathlib import Path
import pickle

# Найти корень репозитория (папка, где есть models/)
ROOT = Path(".").resolve()
while ROOT != ROOT.parent and not (ROOT / "models").exists():
    ROOT = ROOT.parent
assert (ROOT / "models").exists(), f"Не удалось найти корень репозитория от {Path('.').resolve()}"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from safetensors.torch import load_file as load_safetensors

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
    else "cpu"
)
print("ROOT:", ROOT)
print("Device:", device)

ROOT: /Users/vasilij/Documents/ML/proj/ML_project-football-
Device: mps


In [2]:
# Где лежит обученный энкодер (у тебя уже обучен)
# Сейчас ожидаем model.safetensors в этой папке:
ENCODER_CKPT_DIR = ROOT / "mpp_mini_output" / "final_model"

METADATA_DIR = ROOT / "notebooks" / "train_sample_processed" / "metadata"
PROCESSED_CSV = ROOT / "notebooks" / "train_sample_processed" / "processed.csv"
RATINGS_BY_SEASON_CSV = ROOT / "dataset" / "sofifa_ratings_by_season.csv"
OUTPUT_DIR = ROOT / "notebooks" / "rating_head_output" / "next_year"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert ENCODER_CKPT_DIR.exists(), f"Чекпоинт не найден: {ENCODER_CKPT_DIR}"
assert (ENCODER_CKPT_DIR / "model.safetensors").exists(), f"Не найден model.safetensors в {ENCODER_CKPT_DIR}"
assert METADATA_DIR.exists(), f"Метаданные не найдены: {METADATA_DIR}"
assert PROCESSED_CSV.exists(), f"CSV не найден: {PROCESSED_CSV}"
assert RATINGS_BY_SEASON_CSV.exists(), f"Рейтинги по сезонам не найдены: {RATINGS_BY_SEASON_CSV}"

In [3]:
def load_vocab(metadata_dir: Path):
    with open(metadata_dir / "player_name2id.pickle", "rb") as f:
        player_name2id = pickle.load(f)
    with open(metadata_dir / "team_name2id.pickle", "rb") as f:
        team_name2id = pickle.load(f)
    n = len(player_name2id)
    return {
        "player_name2id": player_name2id,
        "team_name2id": team_name2id,
        "player_pad_token_id": n + 1,
        "team_pad_token_id": len(team_name2id),
    }

vocab = load_vocab(METADATA_DIR)
player_pad_token_id = vocab["player_pad_token_id"]
teams_vocab_size = vocab["team_pad_token_id"] + 1
print("player_pad_token_id:", player_pad_token_id)

player_pad_token_id: 6394


In [4]:
import pandas as pd
from data.dataset import MatchDatasetMPP
from data.sofifa import (
    build_aggregated_embeddings_next_year,
    SofifaAggregatedDataset,
)
from models.transformer.encoder import PlayerEncoder
from data.preprocessing import FORM_STATS_SIZE

df = pd.read_csv(PROCESSED_CSV)
match_ds = MatchDatasetMPP(
    df,
    player_name2id=vocab["player_name2id"],
    team_name2id=vocab["team_name2id"],
    max_seq_length=36,
    player_pad_token_id=player_pad_token_id,
    team_pad_token_id=vocab["team_pad_token_id"],
)

ratings_by_season = pd.read_csv(RATINGS_BY_SEASON_CSV)
ratings_by_season = ratings_by_season.dropna(subset=["overall"])
ratings_by_season["rating_year"] = ratings_by_season["rating_year"].astype(int)
print("Строк рейтингов по сезонам (с overall):", len(ratings_by_season))

embed_size = 128
encoder = PlayerEncoder(
    embed_size=embed_size,
    num_layers=1,
    heads=2,
    forward_expansion=4,
    dropout=0.05,
    form_stats_size=FORM_STATS_SIZE,
    players_vocab_size=player_pad_token_id,
    teams_vocab_size=teams_vocab_size,
    positions_vocab_size=25,
    use_teams_embeddings=False,
    position_enc_type="learned",
)
state = load_safetensors(ENCODER_CKPT_DIR / "model.safetensors")
enc_state = {k.replace("encoder.", ""): v for k, v in state.items() if k.startswith("encoder.")}
encoder.load_state_dict(enc_state, strict=True)
for p in encoder.parameters():
    p.requires_grad = False

print("Строим эмбеддинги, агрегированные по (игрок, сезон), таргет — рейтинг на следующий год...")
embeddings, overalls, player_names = build_aggregated_embeddings_next_year(
    encoder, match_ds, df, ratings_by_season, device
)
print("Сэмплов (игрок-сезон с таргетом):", len(overalls))

Строк рейтингов по сезонам (с overall): 6184
Строим эмбеддинги, агрегированные по (игрок, сезон), таргет — рейтинг на следующий год...
Сэмплов (игрок-сезон с таргетом): 6184


In [5]:
meta = pd.DataFrame({"overall": overalls})
dataset = SofifaAggregatedDataset(embeddings, meta)
n = len(dataset)
n_val = max(1, int(n * 0.15))
n_train = n - n_val
train_ds = Subset(dataset, range(0, n_train))
eval_ds = Subset(dataset, range(n_train, n))
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0)
eval_loader = DataLoader(eval_ds, batch_size=64, shuffle=False, num_workers=0)

# Нормализация таргета по train: z = (y - mean) / std
Y_train = np.array(overalls[:n_train], dtype=np.float32)
Y_MEAN = float(Y_train.mean())
Y_STD = float(Y_train.std() + 1e-8)
print("Train:", n_train, "Eval:", n_val)
print(f"Target normalization: mean={Y_MEAN:.4f} std={Y_STD:.4f}")

Train: 5257 Eval: 927


In [6]:
# Головы AdaBoost и XGBoost на тех же эмбеддингах (train)
from sklearn.ensemble import AdaBoostRegressor
from sklearn.metrics import mean_squared_error
import xgboost as xgb

X_train = embeddings[:n_train]
y_train = np.array(overalls[:n_train], dtype=np.float64)
X_eval = embeddings[n_train:n]
y_eval = np.array(overalls[n_train:n], dtype=np.float64)

model_adaboost = AdaBoostRegressor(n_estimators=100, learning_rate=0.5, random_state=42)
model_adaboost.fit(X_train, y_train)
rmse_adaboost = np.sqrt(mean_squared_error(y_eval, model_adaboost.predict(X_eval)))
print("AdaBoost eval RMSE:", round(rmse_adaboost, 4))

model_xgboost = xgb.XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42)
model_xgboost.fit(X_train, y_train)
rmse_xgboost = np.sqrt(mean_squared_error(y_eval, model_xgboost.predict(X_eval)))
print("XGBoost eval RMSE:", round(rmse_xgboost, 4))

AdaBoost eval RMSE: 5.421
XGBoost eval RMSE: 5.2673


In [7]:
from models.heads import RegressionHead
from training.rating_trainer import RatingHeadTrainer

head = RegressionHead(embed_size=embed_size, output_dim=1, hidden_dim=128, pool="per_sequence")
head = head.to(device)

trainer = RatingHeadTrainer(
    head=head,
    train_loader=train_loader,
    eval_loader=eval_loader,
    output_dir=str(OUTPUT_DIR),
    num_epochs=100,
    lr=1e-3,
    weight_decay=0.01,
    device=device,
    logging_steps=10,
    save_best=True,
    target_mean=Y_MEAN,
    target_std=Y_STD,
)
trainer.train()
print("Голова сохранена:", OUTPUT_DIR / "rating_head.pt")
print(f"Denorm params saved in notebook vars: Y_MEAN={Y_MEAN:.4f}, Y_STD={Y_STD:.4f}")

Epoch 1/100  train_loss=2769.3740  eval_rmse=18.3443
Epoch 10/100  train_loss=83.9743  eval_rmse=9.2659
Epoch 20/100  train_loss=51.6192  eval_rmse=7.5940
Epoch 30/100  train_loss=43.8796  eval_rmse=7.2343
Epoch 40/100  train_loss=38.3081  eval_rmse=7.0731
Epoch 50/100  train_loss=34.7589  eval_rmse=6.8154
Epoch 60/100  train_loss=31.3223  eval_rmse=6.7818
Epoch 70/100  train_loss=28.9892  eval_rmse=6.7513
Epoch 80/100  train_loss=27.2320  eval_rmse=6.9295
Epoch 90/100  train_loss=24.4809  eval_rmse=6.5821
Epoch 100/100  train_loss=24.4568  eval_rmse=6.5906
Best eval RMSE: 6.541619930501802
Голова сохранена: /Users/vasilij/Documents/ML/proj/ML_project-football-/notebooks/rating_head_output/next_year/rating_head.pt
Голова сохранена: /Users/vasilij/Documents/ML/proj/ML_project-football-/notebooks/rating_head_output/next_year/rating_head.pt


In [8]:
# (классификатор удалён по просьбе)


In [9]:
# Таблица: игрок — настоящий рейтинг — предсказания всех моделей (нейросеть, AdaBoost, XGBoost)
# Нейросеть предсказывает нормализованный таргет, поэтому денормализуем обратно.
head.eval()
preds_nn = []
with torch.no_grad():
    for idx in range(n_train, n):
        emb = torch.from_numpy(embeddings[idx].copy()).float().unsqueeze(0).unsqueeze(1).to(device)
        mask = torch.ones(1, 1, dtype=torch.long, device=device)
        pred_z = head(emb, mask).squeeze().cpu().item()
        pred = pred_z * Y_STD + Y_MEAN
        preds_nn.append(pred)

preds_adaboost = model_adaboost.predict(X_eval)
preds_xgboost = model_xgboost.predict(X_eval)

table = pd.DataFrame({
    "игрок": player_names[n_train:n],
    "настоящий_рейтинг": overalls[n_train:n],
    "предсказание_нейросеть": preds_nn,
    "предсказание_AdaBoost": preds_adaboost,
    "предсказание_XGBoost": preds_xgboost,
})
table

,игрок,настоящий_рейтинг,предсказание_нейросеть,предсказание_AdaBoost,предсказание_XGBoost
0,Senne Lynen,74.0,84.485138,76.317389,76.698982
1,Seon-Min Moon,71.0,69.966972,73.894612,74.253609
2,Sepp van den Berg,74.0,77.912529,76.983845,79.255142
3,Sereso Geoffroy Gonzaroua Dié,75.0,74.304611,73.393924,73.885429
4,Serge Aurier,81.0,71.611992,75.860257,75.802315
...,...,...,...,...,...
922,Šime Vrsaljko,79.0,81.552200,75.118182,74.473915
923,Šime Vrsaljko,80.0,79.962906,75.696183,74.169167
924,Šime Vrsaljko,81.0,67.881577,77.533638,76.142471
925,Šime Vrsaljko,80.0,77.549568,78.161411,77.262032
